# Genesis — IE 643 | Day 3: Method C (drift control) + continuation LoRA + full evaluation set
**Settings:** Accelerator **GPU T4 x2** · Internet **On** · then **Save Version → Save & Run All (Commit)** (~6–7 h, runs in the background)

| Method | Clip k+1 is made from … |
|---|---|
| `baseline` | text only, hard cut (shot montage) |
| `A_lastframe` | last frame of clip k |
| `B_m2` | last 5 frames of clip k (2 clean latent frames) |
| `C_anchor` | B + colour anchoring to clip 0 + CFG 3.5 on continuation clips |
| `B_m2_lora` | B with the continuation LoRA |
| `C_anchor_lora` | C with the continuation LoRA (**final method**) |

**Prompts are rewritten for a continuous long take**: each prompt describes a change that can actually happen in
~2 seconds of one shot (camera moves, gradual reveals), instead of jumping to a new location every clip.

**GPU plan.** GPU 1 starts generating immediately. GPU 0 first prepares the training data and trains the LoRA
(≤ 2 h), then joins the generation queue. LoRA jobs start once the adapter is saved.

## 1 — Install

In [ ]:
!pip uninstall -y -q torchao
!pip install -q -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg huggingface_hub hf_xet peft pandas
import diffusers, transformers, torch, peft
print("diffusers", diffusers.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__,
      "| torch", torch.__version__, "| GPUs", torch.cuda.device_count())

## 2 — Configuration

In [ ]:
import os, json, time, gc, shutil, re, html, sys, subprocess, threading
import numpy as np
import torch

os.environ["HF_HOME"] = "/tmp/hf_home"
MODEL_ID, WEIGHTS_DIR = "Wan-AI/Wan2.2-TI2V-5B-Diffusers", "/tmp/wan22_ti2v5b"
OUT = "/kaggle/working/genesis3"
for d in ["embeds", "videos", "latents", "frames", "logs", "manifests", "code", "figures", "queue", "train", "lora"]:
    os.makedirs(f"{OUT}/{d}", exist_ok=True)
N_GPU = torch.cuda.device_count()

HEIGHT, WIDTH, NUM_FRAMES, FPS = 480, 832, 49, 24
STEPS, GUIDANCE, FLOW_SHIFT, BASE_SEED, N_CLIPS = 30, 5.0, 3.0, 2000, 5
CFG_CONT, ANCHOR = 3.5, 0.7

NEG_PROMPT = ("Bright tones, overexposed, static, blurred details, subtitles, style, works, paintings, images, static, "
              "overall gray, worst quality, low quality, JPEG compression residue, ugly, incomplete, extra fingers, "
              "poorly drawn hands, poorly drawn faces, deformed, disfigured, misshapen limbs, fused fingers, still picture, "
              "messy background, three legs, many people in the background, walking backwards")

# Continuous-take prompts: each step is a change that fits in ~2 s of one shot.
PROMPT_SEQS = {
    "fox": [
        "A red fox with a bushy white-tipped tail walks slowly through a snowy pine forest at sunrise, soft golden light, steady tracking shot.",
        "The same red fox keeps walking between the pine trees and pauses to sniff the snow, soft golden sunrise light, steady tracking shot.",
        "The same red fox lifts its head, listens, then starts trotting forward through the snowy pine forest, soft golden sunrise light, steady tracking shot.",
        "The same red fox leaps and pounces into the deep snow, digging with its front paws, soft golden sunrise light, steady tracking shot.",
        "The same red fox pulls its head out of the snow and walks on between the pine trees, soft golden sunrise light, steady tracking shot.",
    ],
    "boat": [
        "A small white paper boat floats along a rain-filled street gutter at night, neon shop signs reflecting in the water, close-up tracking shot.",
        "The same small white paper boat bobs and turns slowly as it drifts along the gutter, raindrops rippling around it, neon reflections, close-up tracking shot.",
        "The same small white paper boat speeds up as the water in the gutter flows faster, neon reflections streaking past, close-up tracking shot.",
        "The same small white paper boat slows down and drifts into a calm puddle by the curb, neon reflections, close-up tracking shot.",
        "The same small white paper boat floats still in the calm puddle as the rain softens, neon reflections glowing on the water, close-up shot.",
    ],
    "balloon": [
        "A red and yellow striped hot air balloon floats above a misty green valley at dawn, soft pastel sky, slow aerial shot.",
        "The camera slowly circles the same red and yellow striped hot air balloon as the morning mist below begins to thin, revealing a winding river, soft pastel sky.",
        "The same red and yellow striped hot air balloon drifts over the winding river as the mist clears and sunlight reaches the valley, slow aerial shot.",
        "The burner of the same red and yellow striped hot air balloon flares orange as it rises higher above the sunlit river valley, slow aerial shot.",
        "The same red and yellow striped hot air balloon floats high above the bright green valley in clear morning light, distant hills on the horizon, slow aerial shot.",
    ],
    "car": [
        "A vintage red convertible car drives along a winding coastal cliff road at sunset, the ocean sparkling below, drone shot following the car.",
        "The drone keeps following the same vintage red convertible car along the coastal cliff road; far ahead a small white lighthouse appears on the headland, sunset light.",
        "The same vintage red convertible car continues along the cliff road toward the white lighthouse, waves breaking on the rocks below, sunset light, drone tracking shot.",
        "The same vintage red convertible car drives past the white lighthouse on the headland while the drone slowly rises, sunset light.",
        "The same vintage red convertible car slows down on the road beyond the white lighthouse as the sun touches the ocean horizon, drone tracking shot.",
    ],
}
SEEDS = {s: [BASE_SEED + 100 * j + i for i in range(N_CLIPS)] for j, s in enumerate(PROMPT_SEQS)}
METHODS = {   # name: (m, cfg_cont, anchor, lora)
    "baseline":      (1, None, 0.0, False),
    "A_lastframe":   (1, None, 0.0, False),
    "B_m2":          (2, None, 0.0, False),
    "C_anchor":      (2, CFG_CONT, ANCHOR, False),
    "B_m2_lora":     (2, None, 0.0, True),
    "C_anchor_lora": (2, CFG_CONT, ANCHOR, True),
}
JOB_ORDER = ["baseline", "C_anchor", "B_m2", "A_lastframe", "C_anchor_lora", "B_m2_lora"]   # priority
JOBS = []
for meth in JOB_ORDER:
    m, cfgc, anc, lora = METHODS[meth]
    for s in PROMPT_SEQS:
        deps = [] if meth == "baseline" else [f"baseline_{s}"] + (["lora"] if lora else [])
        JOBS.append({"id": f"{meth}_{s}", "seq": s, "method": meth, "m": m, "cfg_cont": cfgc,
                     "anchor": anc, "lora": lora, "n_clips": N_CLIPS, "seeds": SEEDS[s], "deps": deps})

LORA_DIR = f"{OUT}/lora/genesis_continuation_lora"
TRAIN = {"SELECTION": f"{OUT}/train/selection.json", "ZIP": "/tmp/openvid/OpenVid_part0.zip",
         "CAPTIONS": f"{OUT}/train/caption_embeds.pt", "PREPARED": f"{OUT}/train/latents.pt",
         "H": 384, "W": 672, "FRAMES": 33, "STEPS": 800, "RANK": 32, "LR": 1e-4, "MAX_MIN": 120,
         "SHIFT": 3.0, "M_CHOICES": [1, 2, 3], "M_PROBS": [0.3, 0.4, 0.3], "DROP_TEXT": 0.1, "CKPT_EVERY": 100}
N_TRAIN = 300
CONFIG = {"OUT": OUT, "WEIGHTS_DIR": WEIGHTS_DIR, "EMBEDS": f"{OUT}/embeds/day3_embeds.pt", "HEIGHT": HEIGHT,
          "WIDTH": WIDTH, "NUM_FRAMES": NUM_FRAMES, "STEPS": STEPS, "GUIDANCE": GUIDANCE, "FLOW_SHIFT": FLOW_SHIFT,
          "FPS": FPS, "DEVICE": "cuda", "VAE_OFFLOAD": True, "DTYPE": "float16", "LORA_DIR": LORA_DIR, "TRAIN": TRAIN}
PLAN = f"{OUT}/logs/day3_plan.json"
json.dump({"config": CONFIG, "jobs": JOBS, "prompts": PROMPT_SEQS, "negative_prompt": NEG_PROMPT,
           "methods": {k: dict(zip(["m", "cfg_cont", "anchor", "lora"], v)) for k, v in METHODS.items()}},
          open(PLAN, "w"), indent=2)
print(len(JOBS), "jobs,", sum(1 if j["method"] == "baseline" else 0 for j in JOBS) * N_CLIPS
      + sum(0 if j["method"] == "baseline" else N_CLIPS - 1 for j in JOBS), "clips")

## 3 — Start downloading the LoRA training data in the background
**OpenVid-1M** (CC-BY-4.0, research use): we download one archive (`OpenVid_part0.zip`, ~32 GB, to temporary disk)
plus the caption CSV, and later pick ~300 good clips from it.

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
DL = {"status": "running", "err": None}
def _download():
    try:
        DL["csv"] = hf_hub_download("nkp37/OpenVid-1M", "data/train/OpenVid-1M.csv", repo_type="dataset", local_dir="/tmp/openvid")
        DL["zip"] = hf_hub_download("nkp37/OpenVid-1M", "OpenVid_part0.zip", repo_type="dataset", local_dir="/tmp/openvid")
        if DL["zip"] != TRAIN["ZIP"]:
            TRAIN["ZIP"] = DL["zip"]
        DL["status"] = "done"
    except Exception as e:
        DL["status"], DL["err"] = "failed", repr(e)
dl_thread = threading.Thread(target=_download, daemon=True); dl_thread.start()
print("download started in background")

## 4 — Text encoder: encode the evaluation prompts (the encoder stays loaded for the training captions)

In [ ]:
from transformers import AutoTokenizer, UMT5EncoderModel
import ftfy
snapshot_download(MODEL_ID, local_dir=WEIGHTS_DIR, allow_patterns=["model_index.json", "tokenizer/*", "text_encoder/*"])
tok = AutoTokenizer.from_pretrained(f"{WEIGHTS_DIR}/tokenizer")
if N_GPU >= 2:
    te = UMT5EncoderModel.from_pretrained(f"{WEIGHTS_DIR}/text_encoder", torch_dtype=torch.float32,
                                          device_map="auto", max_memory={0: "14GiB", 1: "14GiB", "cpu": "20GiB"})
    te_device = "cuda:0"
else:
    te = UMT5EncoderModel.from_pretrained(f"{WEIGHTS_DIR}/text_encoder", torch_dtype=torch.bfloat16); te_device = "cpu"
te.eval()

def prompt_clean(t):
    t = ftfy.fix_text(t); t = html.unescape(html.unescape(t)).strip()
    return re.sub(r"\s+", " ", t).strip()

@torch.no_grad()
def t5_tokens(text, max_len=512):
    # Real (unpadded) token embeddings; padding positions are zeros when padded to 512 later — same as diffusers.
    enc = tok([prompt_clean(text)], padding="max_length", max_length=max_len, truncation=True,
              add_special_tokens=True, return_attention_mask=True, return_tensors="pt")
    ids, mask = enc.input_ids.to(te_device), enc.attention_mask.to(te_device)
    n = int(mask.sum())
    return te(ids, mask).last_hidden_state[0, :n].float().cpu()

def pad512(h):
    out = torch.zeros(512, h.shape[-1]); out[: h.shape[0]] = h; return out.unsqueeze(0)

E = {"__negative__": pad512(t5_tokens(NEG_PROMPT)), "__empty__": pad512(t5_tokens(""))}
for s, ps in PROMPT_SEQS.items():
    for i, p in enumerate(ps):
        E[f"{s}_{i}"] = pad512(t5_tokens(p))
torch.save(E, CONFIG["EMBEDS"]); print("eval prompts encoded:", len(E))

## 5 — Pick ~300 training clips from the downloaded archive and encode their captions

In [ ]:
import pandas as pd, zipfile
t0 = time.time()
while DL["status"] == "running" and time.time() - t0 < 50 * 60:    # give up on the LoRA after 50 min of waiting
    time.sleep(30)
if DL["status"] == "running":
    DL["status"], DL["err"] = "failed", "download still running after 50 min"
print(f"download {DL['status']} after waiting {time.time() - t0:.0f}s", DL["err"] or "")
try:
    assert DL["status"] == "done", DL["err"]
    df = pd.read_csv(DL["csv"])
    cols = {c.lower(): c for c in df.columns}
    def col(*subs):
        for c in cols:
            if all(s in c for s in subs):
                return cols[c]
        return None
    vc, cc = col("video"), col("caption")
    aes, mot, tmp, sec = col("aesthetic"), col("motion"), col("temporal"), col("second")
    zf = zipfile.ZipFile(TRAIN["ZIP"])
    names = {os.path.basename(n): n for n in zf.namelist() if n.lower().endswith(".mp4")}
    df["_base"] = df[vc].astype(str).map(os.path.basename)
    d = df[df["_base"].isin(names)].copy()
    print(len(names), "videos in archive,", len(d), "with captions")
    if sec: d = d[d[sec] >= 2.0]
    if aes: d = d[d[aes] >= d[aes].quantile(0.5)]
    if mot:
        lo, hi = d[mot].quantile([0.3, 0.9]); d = d[(d[mot] >= lo) & (d[mot] <= hi)]   # moving, but not chaotic
    if tmp: d = d[d[tmp] >= d[tmp].quantile(0.3)]
    d = d.sample(n=min(N_TRAIN, len(d)), random_state=0)
    sel, caps = [], {"__empty__": t5_tokens("").half()}
    for j, (_, r) in enumerate(d.iterrows()):
        k = f"cap_{j}"
        sel.append({"member": names[r["_base"]], "key": k, "caption": str(r[cc])})
        caps[k] = t5_tokens(str(r[cc])).half()
    json.dump(sel, open(TRAIN["SELECTION"], "w"), indent=1)
    torch.save(caps, TRAIN["CAPTIONS"])
    json.dump(TRAIN, open(PLAN + ".train.json", "w"))
    print(f"selected {len(sel)} training clips; captions encoded")
    # keep the plan's archive path in sync with where the file actually landed
    P = json.load(open(PLAN)); P["config"]["TRAIN"]["ZIP"] = TRAIN["ZIP"]; json.dump(P, open(PLAN, "w"), indent=2)
except Exception as e:
    print("LoRA data preparation FAILED — LoRA jobs will be skipped:", repr(e))
    open(f"{OUT}/queue/lora.failed", "w").write(repr(e))

del te; gc.collect(); torch.cuda.empty_cache()
shutil.rmtree(f"{WEIGHTS_DIR}/text_encoder", ignore_errors=True); shutil.rmtree("/tmp/hf_home", ignore_errors=True)

## 6 — Download transformer, VAE and scheduler

In [ ]:
snapshot_download(MODEL_ID, local_dir=WEIGHTS_DIR, allow_patterns=["model_index.json", "transformer/*", "vae/*", "scheduler/*"])
!du -sh {WEIGHTS_DIR}/* | cat
!df -h /tmp | cat

## 7 — Project code (saved into the output; part of the submission)

In [ ]:
%%writefile /kaggle/working/genesis3/code/genesis_lib.py
"""
genesis_lib.py — Genesis (IE 643): long-form video by chaining short Wan2.2 TI2V-5B clips.

Chaining methods
  baseline : every clip generated independently from text (T2V) and joined with hard cuts.
  A (m=1)  : clip k+1 is conditioned on the LAST FRAME of clip k (the model's own TI2V first-frame conditioning).
  B (m>=2) : clip k+1 is conditioned on the last K = 1 + 4(m-1) FRAMES of clip k. They are re-encoded by the
             causal 3D VAE into m clean latent frames, placed at the start of the new clip's latent; those tokens
             get flow-matching timestep 0 (clean) and are re-injected after every solver step, so the model
             denoises the rest of the clip around a context that shows the motion, not just a still frame.
  C        : Method B + drift control. (i) every generated frame is colour-anchored to clip 0 (its LAB mean and
             standard deviation are pulled toward clip 0's by a factor `anchor`), so exposure and saturation
             cannot creep upward clip after clip; (ii) continuation clips use a lower CFG scale (`cfg_cont`),
             because guidance applied on already-saturated context compounds the drift.
  +LoRA    : any chained method with the continuation LoRA (genesis_train_lora.py) switched on.

Flow matching (what the model was trained with)
  x_s = (1 - s) * x_0 + s * eps,  s in [0, 1]   (s = 1 pure noise, s = 0 clean)
  model predicts velocity v = eps - x_0;  loss = || (eps - x_0) - v_theta(x_s, s, text) ||^2
  TI2V-5B gives every latent token its own timestep: conditioning tokens get s = 0, the rest the current s.
"""
import json, os, time

import numpy as np
import torch
import diffusers
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline, WanPipeline, WanTransformer3DModel
from diffusers.utils import export_to_video
from diffusers.utils.torch_utils import get_module_execution_device, randn_tensor
from diffusers.pipelines.wan.pipeline_wan_i2v import retrieve_latents
from PIL import Image

TEMPORAL = 4      # Wan2.2 VAE temporal compression
TAIL_KEEP = 9     # pixel frames kept losslessly per clip = context for m <= 3


def context_frames(m: int) -> int:
    """Pixel frames the VAE maps to m latent frames (first frame alone, then groups of 4)."""
    return 1 + TEMPORAL * (m - 1)


def new_frames_per_clip(num_frames: int, m: int) -> int:
    """Frames a chained clip adds to the video (its first K frames repeat the context and are dropped)."""
    return num_frames - context_frames(m)


def clips_for_duration(seconds: float, fps: int, num_frames: int, m: int) -> int:
    """Number of clips needed so the stitched video lasts at least `seconds`."""
    need = max(int(round(seconds * fps)), num_frames)
    return 1 + int(np.ceil((need - num_frames) / new_frames_per_clip(num_frames, m)))


def _prepare_latents_ctx(self, image, batch_size, num_channels_latents=48, height=480, width=832,
                         num_frames=49, dtype=None, device=None, generator=None, latents=None, last_image=None):
    """Replacement for WanImageToVideoPipeline.prepare_latents (expand_timesteps / TI2V path).

    If self._genesis_ctx (uint8 array, K x H x W x 3) is set, it is encoded into m = (K-1)/4 + 1 clean latent
    frames and the first m latent frames are masked as 'known' (timestep 0, re-injected every step).
    Otherwise falls back to the stock single-image behaviour.
    """
    ctx = getattr(self, "_genesis_ctx", None)
    if ctx is None:
        return WanImageToVideoPipeline.prepare_latents(self, image, batch_size, num_channels_latents, height, width,
                                                       num_frames, dtype, device, generator, latents, last_image)
    K = ctx.shape[0]
    assert (K - 1) % TEMPORAL == 0, f"context length {K} must be 1 + 4(m-1)"
    assert ctx.shape[1:3] == (height, width), f"context frames {ctx.shape[1:3]} != ({height}, {width})"

    T = (num_frames - 1) // self.vae_scale_factor_temporal + 1
    h, w = height // self.vae_scale_factor_spatial, width // self.vae_scale_factor_spatial
    shape = (batch_size, num_channels_latents, T, h, w)
    if latents is None:
        latents = randn_tensor(shape, generator=generator, device=device, dtype=dtype)
    else:
        latents = latents.to(device=device, dtype=dtype)

    vid = torch.from_numpy(np.ascontiguousarray(ctx)).float().div(127.5).sub(1.0)   # K,H,W,3 in [-1, 1]
    vid = vid.permute(3, 0, 1, 2).unsqueeze(0)                                     # 1,3,K,H,W
    vae_dev = get_module_execution_device(self.vae)
    with torch.no_grad():
        lc = retrieve_latents(self.vae.encode(vid.to(vae_dev, self.vae.dtype)), sample_mode="argmax")
    lc = lc.to(device=device, dtype=dtype)
    mean = torch.tensor(self.vae.config.latents_mean).view(1, -1, 1, 1, 1).to(device, dtype)
    inv_std = 1.0 / torch.tensor(self.vae.config.latents_std).view(1, -1, 1, 1, 1).to(device, dtype)
    lc = (lc - mean) * inv_std

    m = lc.shape[2]
    assert m < T, "context must be shorter than the clip"
    cond = torch.zeros_like(latents)
    cond[:, :, :m] = lc.expand(batch_size, -1, -1, -1, -1)
    mask = torch.ones(1, 1, T, h, w, dtype=dtype, device=device)
    mask[:, :, :m] = 0
    if getattr(self, "_genesis_vae_offload", False):
        self.vae.to("cpu")
        torch.cuda.empty_cache()
    return latents, cond, mask


class _T2V(WanPipeline):
    """Stock T2V pipeline, but always computes on the GPU even while the VAE sits on the CPU."""
    @property
    def _execution_device(self):
        return torch.device(self._genesis_device)


class _CtxI2V(WanImageToVideoPipeline):
    """TI2V pipeline whose conditioning can be a multi-frame context (see _prepare_latents_ctx)."""
    _genesis_ctx = None
    _genesis_vae_offload = False
    prepare_latents = _prepare_latents_ctx

    @property
    def _execution_device(self):
        return torch.device(self._genesis_device)


class Genesis:
    """Holds one TI2V-5B transformer + VAE on one GPU and generates / chains clips."""

    def __init__(self, weights_dir, height=480, width=832, num_frames=49, steps=30, guidance=5.0,
                 flow_shift=3.0, fps=24, device="cuda", vae_offload=True, dtype=torch.float16):
        self.h, self.w, self.nf, self.steps, self.cfg, self.fps = height, width, num_frames, steps, guidance, fps
        self.device, self.vae_offload, self.dtype = device, vae_offload, dtype
        idx = json.load(open(f"{weights_dir}/model_index.json"))
        expand = bool(idx.get("expand_timesteps", False))
        assert expand, "TI2V-5B must load with expand_timesteps=True (per-token timesteps)"

        self.vae = AutoencoderKLWan.from_pretrained(f"{weights_dir}/vae", torch_dtype=torch.float32)
        self.vae.enable_tiling()
        self.vae.to("cpu" if vae_offload else device)
        try:
            self.transformer = WanTransformer3DModel.from_pretrained(
                f"{weights_dir}/transformer", torch_dtype=dtype, device_map=device)
        except Exception as e:  # older/newer API without device_map support for this model
            print("device_map load failed, loading on CPU first:", repr(e)[:200])
            self.transformer = WanTransformer3DModel.from_pretrained(
                f"{weights_dir}/transformer", torch_dtype=dtype).to(device)
        self.transformer.eval()

        cfg = json.load(open(f"{weights_dir}/scheduler/scheduler_config.json"))
        cls = getattr(diffusers, cfg["_class_name"])
        sched = cls.from_pretrained(f"{weights_dir}/scheduler")
        key = "flow_shift" if "flow_shift" in sched.config else ("shift" if "shift" in sched.config else None)
        if key:
            sched = cls.from_config(sched.config, **{key: flow_shift})

        common = dict(tokenizer=None, text_encoder=None, vae=self.vae, scheduler=sched,
                      transformer=self.transformer, expand_timesteps=expand)
        self.t2v = _T2V(**common)
        self.i2v = _CtxI2V(**common)
        self.i2v._genesis_vae_offload = vae_offload
        for p in (self.t2v, self.i2v):
            p._genesis_device = device
            p.set_progress_bar_config(disable=True)
        self._lora_path = None      # adapter loaded into the transformer (None = never loaded)
        self.lora_active = False

    # ---------------------------------------------------------------- LoRA
    def set_lora(self, path):
        """path=None switches the adapter off; a directory with lora.safetensors + lora_config.json switches it on."""
        if path is None:
            if self._lora_path is not None and self.lora_active:
                self.transformer.disable_adapters()
            self.lora_active = False
            return
        if self._lora_path is None:
            from peft import LoraConfig, set_peft_model_state_dict
            from safetensors.torch import load_file
            cfg = json.load(open(f"{path}/lora_config.json"))
            self.transformer.add_adapter(LoraConfig(**cfg), adapter_name="genesis")
            sd = load_file(f"{path}/lora.safetensors")
            res = set_peft_model_state_dict(self.transformer, sd, adapter_name="genesis")
            missing = [k for k in getattr(res, "unexpected_keys", [])]
            assert not missing, f"LoRA keys not loaded: {missing[:5]}"
            for n, prm in self.transformer.named_parameters():   # cast only the adapter: some base modules
                if "lora_" in n:                                    # (norms, time embedder) must stay fp32
                    prm.data = prm.data.to(self.dtype)
            self._lora_path = path
        elif path != self._lora_path:
            raise ValueError("a different LoRA is already loaded in this process")
        self.transformer.enable_adapters()
        self.lora_active = True

    # ---------------------------------------------------------------- helpers
    def _emb(self, E, key):
        return E[key].to(self.device, self.dtype)

    def _kwargs(self, E, seed, guidance=None):
        return dict(negative_prompt_embeds=self._emb(E, "__negative__"), height=self.h, width=self.w,
                    num_frames=self.nf, num_inference_steps=self.steps,
                    guidance_scale=self.cfg if guidance is None else guidance,
                    generator=torch.Generator("cpu").manual_seed(int(seed)), output_type="latent")

    @torch.no_grad()
    def _decode(self, lat):
        v = self.vae
        v.to(self.device)
        try:
            z = lat.to(self.device, v.dtype)
            mean = torch.tensor(v.config.latents_mean).view(1, -1, 1, 1, 1).to(z.device, z.dtype)
            std = torch.tensor(v.config.latents_std).view(1, -1, 1, 1, 1).to(z.device, z.dtype)
            video = v.decode(z * std + mean, return_dict=False)[0]
            return self.t2v.video_processor.postprocess_video(video, output_type="np")[0]  # F,H,W,3 in [0,1]
        finally:
            if self.vae_offload:
                v.to("cpu")
            torch.cuda.empty_cache()

    def decode(self, lat):
        try:
            return self._decode(lat)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            self.transformer.to("cpu")
            torch.cuda.empty_cache()
            try:
                return self._decode(lat)
            finally:
                self.transformer.to(self.device)

    # ---------------------------------------------------------------- generation
    def text_clip(self, E, key, seed):
        """Independent clip from text only (baseline, and clip 0 of every chain)."""
        lat = self.t2v(prompt_embeds=self._emb(E, key), **self._kwargs(E, seed)).frames
        return self.decode(lat), lat

    def context_clip(self, ctx_uint8, E, key, seed, guidance=None):
        """Clip conditioned on K = 1 + 4(m-1) previous frames (m = 1 is Method A, m >= 2 is Method B)."""
        if self.vae_offload:
            self.vae.to(self.device)
        self.i2v._genesis_ctx = ctx_uint8
        try:
            dummy = Image.fromarray(ctx_uint8[-1])   # required argument; real conditioning is _genesis_ctx
            lat = self.i2v(image=dummy, prompt_embeds=self._emb(E, key), **self._kwargs(E, seed, guidance)).frames
        finally:
            self.i2v._genesis_ctx = None
            if self.vae_offload:
                self.vae.to("cpu")
        return self.decode(lat), lat


# -------------------------------------------------------------------- storage
def to_uint8(frames):
    return (np.clip(frames, 0.0, 1.0) * 255.0).round().astype(np.uint8)


def frames_ok(frames):
    return bool(np.isfinite(frames).all() and frames.max() > 0.02)


def save_clip(out, tag, frames, lat, fps):
    export_to_video(list(frames), f"{out}/videos/{tag}.mp4", fps=fps)
    torch.save(lat.detach().cpu().half(), f"{out}/latents/{tag}.pt")
    np.save(f"{out}/frames/{tag}_tail.npy", to_uint8(frames[-TAIL_KEEP:]))


def clip_done(out, tag):
    return os.path.exists(f"{out}/frames/{tag}_tail.npy") and os.path.exists(f"{out}/videos/{tag}.mp4")


def load_tail(out, tag):
    return np.load(f"{out}/frames/{tag}_tail.npy")


def lab_stats(frames_u8):
    """Mean and std of each LAB channel over a stack of uint8 RGB frames (OpenCV LAB, 0-255 scale)."""
    import cv2
    lab = np.stack([cv2.cvtColor(f, cv2.COLOR_RGB2LAB) for f in frames_u8]).reshape(-1, 3).astype(np.float32)
    return lab.mean(0), lab.std(0) + 1e-6


def colour_anchor(frames, ref, strength):
    """Pull each frame's LAB mean/std toward the reference stats by `strength` in [0, 1]. frames: float [0,1]."""
    import cv2
    ref_mu, ref_sd = ref
    out = np.empty_like(frames)
    for j, f in enumerate(to_uint8(frames)):
        lab = cv2.cvtColor(f, cv2.COLOR_RGB2LAB).astype(np.float32)
        mu, sd = lab.reshape(-1, 3).mean(0), lab.reshape(-1, 3).std(0) + 1e-6
        t_mu, t_sd = mu + strength * (ref_mu - mu), sd + strength * (ref_sd - sd)
        lab = (lab - mu) / sd * t_sd + t_mu
        rgb = cv2.cvtColor(np.clip(lab, 0, 255).astype(np.uint8), cv2.COLOR_LAB2RGB)
        out[j] = rgb.astype(np.float32) / 255.0
    return out


def read_video(path):
    import imageio.v3 as iio
    return np.stack(list(iio.imiter(path))).astype(np.float32) / 255.0


def log_jsonl(path, rec):
    with open(path, "a") as f:
        f.write(json.dumps(rec) + "\n")


# -------------------------------------------------------------------- one sequence, one method
def run_sequence(G, E, out, seq, n_clips, seeds, method, m, log_path, cfg_cont=None, anchor=0.0, lora=None):
    """method: 'baseline' or a chain name; m: context latent frames (1 = Method A);
    cfg_cont: CFG scale for continuation clips (None = same as clip 0); anchor: colour-anchoring strength;
    lora: LoRA directory to switch on for the continuation clips (None = off)."""
    clips = []
    ref = None
    for i in range(n_clips):
        if method == "baseline" or i == 0:
            tag = f"{seq}_baseline_c{i}"
        else:
            tag = f"{seq}_{method}_c{i}"
        if clip_done(out, tag):
            frames = read_video(f"{out}/videos/{tag}.mp4")
            stats = {"resumed": True}
        else:
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            t0 = time.time()
            if method == "baseline" or i == 0:
                G.set_lora(None)                     # clip 0 is always the shared, unmodified baseline clip
                frames, lat = G.text_clip(E, f"{seq}_{i}", seeds[i])
            else:
                G.set_lora(lora)
                K = context_frames(m)
                ctx = load_tail(out, prev_tag)[-K:]
                frames, lat = G.context_clip(ctx, E, f"{seq}_{i}", seeds[i], guidance=cfg_cont)
                if anchor > 0:
                    if ref is None:
                        ref = lab_stats(to_uint8(read_video(f"{out}/videos/{seq}_baseline_c0.mp4")))
                    frames = np.concatenate([frames[:K], colour_anchor(frames[K:], ref, anchor)], axis=0)
            stats = {"sec": round(time.time() - t0, 1),
                     "peak_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2) if torch.cuda.is_available() else 0.0,
                     "frames_ok": frames_ok(frames)}
            save_clip(out, tag, frames, lat, G.fps)
        drop = 0 if (method == "baseline" or i == 0) else context_frames(m)
        clips.append(frames[drop:])
        prev_tag = tag
        rec = {"seq": seq, "method": method, "m": m, "clip": i, "tag": tag, "cfg_cont": cfg_cont,
               "anchor": anchor, "lora": bool(lora) and i > 0, **stats}
        log_jsonl(log_path, rec)
        print(rec, flush=True)

    full = np.concatenate(clips, axis=0)
    export_to_video(list(full), f"{out}/videos/{seq}_{method}_FULL.mp4", fps=G.fps)
    lengths = [int(c.shape[0]) for c in clips]
    G.set_lora(None)
    json.dump({"seq": seq, "method": method, "m": m, "cfg_cont": cfg_cont, "anchor": anchor, "lora": bool(lora),
               "clip_lengths": lengths,
               "boundaries": [int(b) for b in np.cumsum(lengths)[:-1]], "seconds": round(full.shape[0] / G.fps, 2)},
              open(f"{out}/manifests/{seq}_{method}.json", "w"), indent=2)
    return full

In [ ]:
%%writefile /kaggle/working/genesis3/code/genesis_queue_worker.py
"""genesis_queue_worker.py — one process per GPU; pulls (sequence, method) jobs from a shared file queue.

A job runs once its dependencies are done ('base_<seq>' for chains, 'lora' for LoRA jobs). Jobs are claimed
atomically with O_EXCL lock files, so two workers never run the same job. Finished clips are skipped on re-run.
"""
import argparse, json, os, sys, time, traceback
import torch

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import genesis_lib as gl

ap = argparse.ArgumentParser()
ap.add_argument("--plan", required=True)
ap.add_argument("--worker", required=True)
a = ap.parse_args()

plan = json.load(open(a.plan))
C = plan["config"]
OUT, Q = C["OUT"], f'{C["OUT"]}/queue'
os.makedirs(Q, exist_ok=True)


def st(name, kind):
    return os.path.exists(f"{Q}/{name}.{kind}")


def dep_state(d):
    if st(d, "done"):
        return "ok"
    if st(d, "failed") or st(d, "skipped"):
        return "dead"
    return "wait"


def next_job():
    """Returns (job, None) to run, (None, 'wait') if something may become runnable, (None, 'exit') otherwise."""
    waiting = False
    for job in plan["jobs"]:
        j = job["id"]
        if any(st(j, k) for k in ("done", "failed", "skipped", "claimed")):
            continue
        deps = [dep_state(d) for d in job.get("deps", [])]
        if "dead" in deps:
            open(f"{Q}/{j}.skipped", "w").write("dependency failed")
            continue
        if "wait" in deps:
            waiting = True
            continue
        try:
            fd = os.open(f"{Q}/{j}.claimed", os.O_CREAT | os.O_EXCL | os.O_WRONLY)
            os.write(fd, a.worker.encode()); os.close(fd)
            return job, None
        except FileExistsError:
            continue
    return None, ("wait" if waiting else "exit")


G = gl.Genesis(C["WEIGHTS_DIR"], C["HEIGHT"], C["WIDTH"], C["NUM_FRAMES"], C["STEPS"], C["GUIDANCE"],
               C["FLOW_SHIFT"], C["FPS"], device=C.get("DEVICE", "cuda"), vae_offload=C.get("VAE_OFFLOAD", True),
               dtype=getattr(torch, C.get("DTYPE", "float16")))
open(f"{OUT}/logs/worker{a.worker}.ready", "w").write("ok")
print(f"worker {a.worker} loaded", flush=True)
E = torch.load(C["EMBEDS"])
log = f"{OUT}/logs/runs_worker{a.worker}.jsonl"

while True:
    job, why = next_job()
    if job is None:
        if why == "exit":
            break
        time.sleep(60)
        continue
    print(f"worker {a.worker} -> {job['id']}", flush=True)
    try:
        lora = C["LORA_DIR"] if job.get("lora") else None
        gl.run_sequence(G, E, OUT, job["seq"], job["n_clips"], job["seeds"], job["method"], job["m"], log,
                        cfg_cont=job.get("cfg_cont"), anchor=job.get("anchor", 0.0), lora=lora)
        open(f"{Q}/{job['id']}.done", "w").write(a.worker)
    except Exception:
        tb = traceback.format_exc()
        print(tb, flush=True)
        open(f"{Q}/{job['id']}.failed", "w").write(tb)
        G.set_lora(None)
        torch.cuda.empty_cache()
open(f"{OUT}/logs/worker{a.worker}.done", "w").write("ok")
print(f"worker {a.worker} done", flush=True)

In [ ]:
%%writefile /kaggle/working/genesis3/code/genesis_train_lora.py
"""genesis_train_lora.py — continuation LoRA for Wan2.2 TI2V-5B (Genesis, IE 643).

Goal: TI2V-5B was trained to condition on ONE clean first frame. Method B hands it m = 1..3 clean latent frames.
This LoRA teaches the model to continue real videos from a multi-frame context.

Data: real clips (OpenVid-1M, CC-BY-4.0) -> 33 frames at ~24 fps, 384x672 -> VAE latents (9 latent frames).
Objective (flow matching, per-token timesteps, loss only on the frames to be generated):
    m ~ {1,2,3};  sigma ~ shift(logit-normal);  eps ~ N(0, I)
    x_sigma = (1 - sigma) x0 + sigma eps   on frames m..T-1;   frames 0..m-1 stay clean (x0), timestep 0
    loss = || v_theta(x, t, text) - (eps - x0) ||^2   averaged over frames m..T-1 only
Base weights frozen (fp16); LoRA (rank r) on the attention projections of every block, trained in fp32.
"""
import argparse, json, math, os, random, sys, tempfile, time, traceback, zipfile

import numpy as np
import torch

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

ap = argparse.ArgumentParser()
ap.add_argument("--plan", required=True)
a = ap.parse_args()
plan = json.load(open(a.plan))
C = plan["config"]
T = C["TRAIN"]
OUT, Q, LORA_DIR = C["OUT"], f'{C["OUT"]}/queue', C["LORA_DIR"]
DEV = C.get("DEVICE", "cuda")
DTYPE = getattr(torch, C.get("DTYPE", "float16"))
os.makedirs(Q, exist_ok=True); os.makedirs(LORA_DIR, exist_ok=True)
LOG = f"{OUT}/logs/train_lora.jsonl"


def log(rec):
    print(rec, flush=True)
    with open(LOG, "a") as f:
        f.write(json.dumps(rec) + "\n")


# ------------------------------------------------------------------ data preparation (real clips -> latents)
def load_clip(path, n, H, W, target_fps=24):
    import cv2
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
    stride = max(1, int(round(fps / target_fps)))
    frames, idx = [], 0
    while len(frames) < n:
        ok, fr = cap.read()
        if not ok:
            break
        if idx % stride == 0:
            h, w = fr.shape[:2]
            s = max(H / h, W / w)
            fr = cv2.resize(fr, (math.ceil(w * s), math.ceil(h * s)), interpolation=cv2.INTER_AREA)
            y, x = (fr.shape[0] - H) // 2, (fr.shape[1] - W) // 2
            frames.append(cv2.cvtColor(fr[y:y + H, x:x + W], cv2.COLOR_BGR2RGB))
        idx += 1
    cap.release()
    return np.stack(frames) if len(frames) == n else None


def prepare():
    from diffusers import AutoencoderKLWan
    from diffusers.pipelines.wan.pipeline_wan_i2v import retrieve_latents
    sel = json.load(open(T["SELECTION"]))
    vae = AutoencoderKLWan.from_pretrained(f'{C["WEIGHTS_DIR"]}/vae', torch_dtype=torch.float32).to(DEV)
    vae.enable_tiling()
    mean = torch.tensor(vae.config.latents_mean).view(1, -1, 1, 1, 1).to(DEV)
    inv_std = 1.0 / torch.tensor(vae.config.latents_std).view(1, -1, 1, 1, 1).to(DEV)
    keys, lats, t0 = [], [], time.time()
    zf = zipfile.ZipFile(T["ZIP"]) if T.get("ZIP") else None
    tmp = tempfile.mkdtemp()
    for i, r in enumerate(sel):
        try:
            if zf is not None:
                path = zf.extract(r["member"], tmp)
            else:
                path = r["path"]
            fr = load_clip(path, T["FRAMES"], T["H"], T["W"])
            if zf is not None:
                os.remove(path)
            if fr is None:
                continue
            vid = torch.from_numpy(fr).float().div(127.5).sub(1).permute(3, 0, 1, 2).unsqueeze(0).to(DEV)
            with torch.no_grad():
                z = retrieve_latents(vae.encode(vid), sample_mode="argmax")
            lats.append(((z - mean) * inv_std).half().cpu()); keys.append(r["key"])
        except Exception as e:
            print("skip", r.get("member", r.get("path")), repr(e)[:120], flush=True)
        if i % 25 == 0:
            print(f"prep {i}/{len(sel)} kept {len(lats)} ({time.time() - t0:.0f}s)", flush=True)
    torch.save({"keys": keys, "latents": lats}, T["PREPARED"])
    log({"phase": "prep", "kept": len(lats), "of": len(sel), "sec": round(time.time() - t0)})
    del vae; torch.cuda.empty_cache()


# ------------------------------------------------------------------ training
def train():
    from diffusers import WanTransformer3DModel
    from peft import LoraConfig, get_peft_model_state_dict, set_peft_model_state_dict
    from safetensors.torch import load_file, save_file

    data = torch.load(T["PREPARED"])
    caps = torch.load(T["CAPTIONS"])       # key -> [n_tokens, 4096] fp16 (unpadded)
    keys, lats = data["keys"], data["latents"]
    assert len(lats) > 10, f"too few training clips ({len(lats)})"

    def text(key):
        e = caps[key].float()
        out = torch.zeros(512, e.shape[-1]); out[: e.shape[0]] = e[:512]
        return out.unsqueeze(0).to(DEV, DTYPE)

    try:
        tr = WanTransformer3DModel.from_pretrained(f'{C["WEIGHTS_DIR"]}/transformer', torch_dtype=DTYPE, device_map=DEV)
    except Exception:
        tr = WanTransformer3DModel.from_pretrained(f'{C["WEIGHTS_DIR"]}/transformer', torch_dtype=DTYPE).to(DEV)
    tr.requires_grad_(False)
    if DEV == "cuda":
        tr.enable_gradient_checkpointing()
    lcfg = {"r": T["RANK"], "lora_alpha": T["RANK"], "init_lora_weights": True,
            "target_modules": ["to_q", "to_k", "to_v", "to_out.0"]}
    tr.add_adapter(LoraConfig(**lcfg), adapter_name="genesis")
    params = []
    for n, p in tr.named_parameters():
        if "lora_" in n:
            p.data = p.data.float(); p.requires_grad_(True); params.append(p)
    n_lora = sum(p.numel() for p in params)
    json.dump(lcfg, open(f"{LORA_DIR}/lora_config.json", "w"), indent=2)

    start = 0
    state_p = f"{LORA_DIR}/state.json"
    if os.path.exists(state_p) and os.path.exists(f"{LORA_DIR}/lora.safetensors"):
        set_peft_model_state_dict(tr, load_file(f"{LORA_DIR}/lora.safetensors"), adapter_name="genesis")
        for p in params:
            p.data = p.data.float()
        start = json.load(open(state_p))["step"]
        print("resumed LoRA at step", start, flush=True)

    opt = torch.optim.AdamW(params, lr=T["LR"], weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEV == "cuda" and DTYPE == torch.float16))
    rng = random.Random(1234 + start)
    gen = torch.Generator().manual_seed(1234 + start)
    shift = T["SHIFT"]

    def save(step):
        sd = {k: v.detach().float().cpu().contiguous() for k, v in get_peft_model_state_dict(tr, adapter_name="genesis").items()}
        save_file(sd, f"{LORA_DIR}/lora.safetensors")
        json.dump({"step": step}, open(state_p, "w"))

    log({"phase": "train_start", "clips": len(lats), "lora_params_M": round(n_lora / 1e6, 2), "start": start})
    t0, losses = time.time(), []
    tr.train()
    done = start
    for step in range(start, T["STEPS"]):
        if (time.time() - t0) / 60 > T["MAX_MIN"]:
            log({"phase": "time_limit", "step": step}); break
        i = rng.randrange(len(lats))
        x0 = lats[i].to(DEV, torch.float32)                       # 1,48,T,h,w
        Tl, h, w = x0.shape[2:]
        m = rng.choices(T["M_CHOICES"], weights=T["M_PROBS"])[0]
        m = min(m, Tl - 1)
        u = torch.sigmoid(torch.randn(1, generator=gen)).item()  # logit-normal
        sigma = shift * u / (1 + (shift - 1) * u)
        eps = torch.randn(x0.shape, generator=gen).to(DEV)
        mask = torch.ones(1, 1, Tl, h, w, device=DEV); mask[:, :, :m] = 0
        xt = (1 - sigma) * x0 + sigma * eps
        inp = (1 - mask) * x0 + mask * xt
        ts = (mask[0, 0][:, ::2, ::2] * sigma * 1000.0).flatten().unsqueeze(0)
        emb = text("__empty__" if rng.random() < T["DROP_TEXT"] else keys[i])
        pred = tr(hidden_states=inp.to(DTYPE), timestep=ts,
                  encoder_hidden_states=emb, return_dict=False)[0].float()
        target = eps - x0
        loss = (((pred - target) ** 2) * mask).sum() / (mask.sum() * x0.shape[1])
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update()
        losses.append(loss.item())
        done = step + 1
        if (step + 1) % 25 == 0:
            el = time.time() - t0
            log({"step": step + 1, "loss": round(float(np.mean(losses[-25:])), 4), "m": m,
                 "sec_per_step": round(el / (step + 1 - start), 2), "min": round(el / 60, 1)})
        if (step + 1) % T["CKPT_EVERY"] == 0:
            save(step + 1)
    save(done)
    log({"phase": "train_end", "steps_done": done, "min": round((time.time() - t0) / 60, 1),
         "final_loss_avg100": round(float(np.mean(losses[-100:])), 4) if losses else None})


try:
    if not os.path.exists(T["PREPARED"]):
        prepare()
    if os.path.exists(f"{LORA_DIR}/state.json") and json.load(open(f"{LORA_DIR}/state.json"))["step"] >= T["STEPS"]:
        print("LoRA already trained", flush=True)
    else:
        train()
    open(f"{Q}/lora.done", "w").write("ok")
except Exception:
    tb = traceback.format_exc()
    print(tb, flush=True)
    open(f"{Q}/lora.failed", "w").write(tb)
    sys.exit(1)

## 8 — Run: GPU 1 generates right away; GPU 0 trains the LoRA, then generates (~6 h)
Finished clips are saved immediately and skipped if the notebook is re-run.

In [ ]:
env = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", TOKENIZERS_PARALLELISM="false")
for f in os.listdir(f"{OUT}/queue"):               # stale claims from an interrupted earlier run
    if f.endswith(".claimed") and not os.path.exists(f"{OUT}/queue/{f[:-8]}.done"):
        os.remove(f"{OUT}/queue/{f}")

def launch(script, gpu, tag, extra=()):
    logf = open(f"{OUT}/logs/{tag}.log", "a")
    return subprocess.Popen([sys.executable, f"{OUT}/code/{script}", "--plan", PLAN, *extra],
                            env=dict(env, CUDA_VISIBLE_DEVICES=str(gpu)), stdout=logf, stderr=subprocess.STDOUT)

def wait_ready(p, w):
    while not os.path.exists(f"{OUT}/logs/worker{w}.ready") and p.poll() is None:
        time.sleep(15)

procs = {}
if N_GPU >= 2:
    procs["worker1"] = launch("genesis_queue_worker.py", 1, "worker1", ["--worker", "1"]); wait_ready(procs["worker1"], "1")
lora_needed = not os.path.exists(f"{OUT}/queue/lora.failed") and not os.path.exists(f"{OUT}/queue/lora.done")
procs["train"] = launch("genesis_train_lora.py", 0, "train") if lora_needed else None
t0, worker0 = time.time(), None
while True:
    tr = procs["train"]
    if worker0 is None and (tr is None or tr.poll() is not None):
        if tr is not None and not os.path.exists(f"{OUT}/queue/lora.done"):
            open(f"{OUT}/queue/lora.failed", "a").write(f"train exit code {tr.returncode}")
            print(open(f"{OUT}/logs/train.log").read()[-4000:])
        worker0 = procs["worker0"] = launch("genesis_queue_worker.py", 0, "worker0", ["--worker", "0"])
    running = [k for k, p in procs.items() if p is not None and p.poll() is None]
    if worker0 is not None and not running:
        break
    time.sleep(300)
    msg = []
    for k in procs:
        if procs[k] is None: continue
        lines = open(f"{OUT}/logs/{k}.log").read().strip().splitlines()
        msg.append(f"{k}: {lines[-1][:140] if lines else '-'}")
    print(f"[{(time.time() - t0) / 60:5.0f} min] " + " | ".join(msg), flush=True)
for k, p in procs.items():
    if p is not None:
        print(k, "exit code", p.returncode)
print({k: len([f for f in os.listdir(f'{OUT}/queue') if f.endswith(k)]) for k in ['.done', '.failed', '.skipped']})

## 9 — Qualitative outputs (seam grids, 2×3 comparison videos, run summary)

In [ ]:
sys.path.insert(0, f"{OUT}/code")
import genesis_lib as gl
from PIL import Image, ImageDraw
from diffusers.utils import export_to_video

def label(img, text):
    im = Image.fromarray(img); d = ImageDraw.Draw(im)
    d.rectangle([0, 0, 8 + 7 * len(text), 18], fill=(0, 0, 0)); d.text((4, 3), text, fill=(255, 255, 255))
    return np.asarray(im)
def small(f, s=2):
    return np.asarray(Image.fromarray(gl.to_uint8(f)).resize((WIDTH // s, HEIGHT // s)))

ORDER = ["baseline", "A_lastframe", "B_m2", "C_anchor", "B_m2_lora", "C_anchor_lora"]
for seq in PROMPT_SEQS:
    vids, rows = {}, []
    for meth in ORDER:
        mp, vp = f"{OUT}/manifests/{seq}_{meth}.json", f"{OUT}/videos/{seq}_{meth}_FULL.mp4"
        if not (os.path.exists(mp) and os.path.exists(vp)):
            continue
        man, v = json.load(open(mp)), gl.read_video(vp); vids[meth] = v
        for b in man["boundaries"]:
            rows.append(np.concatenate([label(small(v[j], 3), f"{meth} seam@{b} f{j}") for j in (b - 2, b - 1, b, b + 1)], axis=1))
    if rows:
        Image.fromarray(np.concatenate(rows, axis=0)).save(f"{OUT}/figures/{seq}_seams.jpg", quality=88)
    if vids:
        L, blank = max(v.shape[0] for v in vids.values()), np.zeros((HEIGHT // 2, WIDTH // 2, 3), np.uint8)
        frames = []
        for t in range(L):
            tiles = [label(small(vids[k][min(t, vids[k].shape[0] - 1)]), k) if k in vids else blank for k in ORDER]
            frames.append(np.concatenate([np.concatenate(tiles[:3], 1), np.concatenate(tiles[3:], 1)], 0).astype(np.float32) / 255)
        export_to_video(frames, f"{OUT}/videos/{seq}_COMPARE.mp4", fps=FPS)
    print(seq, {k: f"{v.shape[0] / FPS:.1f}s" for k, v in vids.items()})

recs = []
for f in os.listdir(f"{OUT}/logs"):
    if f.startswith("runs_worker"):
        recs += [json.loads(l) for l in open(f"{OUT}/logs/{f}") if l.strip()]
new = [r for r in recs if "sec" in r]
if new:
    print(f"\n{len(new)} clips generated | mean {np.mean([r['sec'] for r in new]):.0f}s/clip | max peak "
          f"{max(r['peak_gb'] for r in new):.2f} GB | black/NaN: {[r['tag'] for r in new if not r.get('frames_ok', True)] or 'none'}")
if os.path.exists(f"{OUT}/logs/train_lora.jsonl"):
    tl = [json.loads(l) for l in open(f"{OUT}/logs/train_lora.jsonl")]
    print("LoRA training:", [r for r in tl if "phase" in r])
json.dump(recs, open(f"{OUT}/logs/day3_runs_all.json", "w"), indent=1)
for f in os.listdir(f"{OUT}/queue"):
    if f.endswith(".failed"):
        print("FAILED:", f, open(f"{OUT}/queue/{f}").read()[-1500:])
!du -sh {OUT} | cat

## What to send back
- `figures/*_seams.jpg`, and your impression of `videos/*_COMPARE.mp4` (2×3 grid: top row baseline / A / B, bottom row C / B+LoRA / C+LoRA)
- the summary lines printed by step 9 (clips, s/clip, peak GB, LoRA training lines, any FAILED)

Next: the metrics notebook (prompt adherence, subject/background consistency, seam smoothness, colour drift,
motion) over the Day 2 and Day 3 outputs, then the Gradio interface.